In [149]:
import struct
from PIL import Image, ImageDraw

data = open('../COMM.IMG', 'rb').read()
print(len(data))

row_width = 896
# right_pixels = 192   
right_pixels = 130  
# right_pixels = int(right_bytes / 2)
right_bytes = right_pixels * 2
left_bytes = row_width - right_bytes    
left_pixels = int(left_bytes * 2)
print(f"left_bytes: {left_bytes}, left_pixels: {left_pixels}")
print(f"right_bytes: {right_bytes}, right_pixels: {right_pixels}")

img = Image.new('RGB', (left_pixels + right_pixels, 512), (0, 0, 0))
pix = img.load()

# 320 x 240
w = 320
# w = 192
h = 240
img2 = Image.new('RGB', (w, h), (0, 0, 0))
pix2 = img2.load()

for y in range(512): # byte
    # Left: 4bpp indexed (grayscale without runtime CLUT)
    for xb in range(left_bytes):
        offset = y * row_width + xb
        byte = data[offset]
        lo = (byte & 0x0F) * 17
        hi = ((byte >> 4) & 0x0F) * 17
        pix[xb * 2, y] = (lo, lo, lo)
        pix[xb * 2 + 1, y] = (hi, hi, hi)

    # Right: 16-bit RGB555
    for x in range(right_pixels):
        offset = y * row_width + left_bytes + x * 2
        val = struct.unpack_from('<H', data, offset)[0]
        r = (val & 0x1F) << 3
        g = ((val >> 5) & 0x1F) << 3
        b = ((val >> 10) & 0x1F) << 3
        pix[left_pixels + x, y] = (r, g, b)

x_cut = 192
left_bytes = row_width - x_cut * 2
for y in range(h):
    for x in range(w):
        if x < x_cut:
            offset = y * row_width + left_bytes + x * 2
            val = struct.unpack_from('<H', data, offset)[0]
            r = (val & 0x1F) << 3
            g = ((val >> 5) & 0x1F) << 3
            b = ((val >> 10) & 0x1F) << 3
            pix2[x, y] = (r, g, b)
        else:
            offset = (y+h+16) * row_width + left_bytes + (x-x_cut+64) * 2
            # offset = (y) * row_width + left_bytes + x * 2
            val = struct.unpack_from('<H', data, offset)[0]
            r = (val & 0x1F) << 3
            g = ((val >> 5) & 0x1F) << 3
            b = ((val >> 10) & 0x1F) << 3
            pix2[x, y] = (r, g, b)

# Draw bounding boxes for left and right regions
draw = ImageDraw.Draw(img)

# Left region box (4bpp): red
draw.rectangle([0, 0, left_pixels - 1, 511], outline='red', width=2)
draw.text((4, 4), f"4bpp ({left_pixels}x512)", fill='red')

# Right region box (16-bit): cyan
draw.rectangle([left_pixels, 0, left_pixels + right_pixels - 1, 511], outline='cyan', width=2)
draw.text((left_pixels + 4, 4), f"16-bit RGB555 ({right_pixels}x512)", fill='cyan')

img.show()
img2.show()

458752
left_bytes: 636, left_pixels: 1272
right_bytes: 260, right_pixels: 130
